In [ ]:
# Montar Google Drive e instalar bibliotecas

from google.colab import drive
drive.mount('/content/drive')

%pip install -q transformers python-docx

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path
from docx import Document
import re
import os

In [ ]:
#Carregamento do Modelo de Classificação

modelo_nome = "fabiopassos/misobr-bertimbau-base"
# modelo_nome = "hugo-guigo/bertimbau-misoginia-ptbr"
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
modelo = AutoModelForSequenceClassification.from_pretrained(modelo_nome)
modelo.eval()
print("Modelo carregado com sucesso!")

In [ ]:
# Configurando dispositivo para aceleração por hardware (GPU se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo.to(device)
print(f"Modelo configurado para executar em: {device}")

In [ ]:
def extrair_musicas_de_diretorio(caminho_diretorio):
    """
    Varre todos os arquivos .docx de uma pasta e extrai apenas as letras das músicas como strings.
    """
    diretorio = Path(caminho_diretorio)
    todas_musicas = []

    if not diretorio.exists():
        print(f"Aviso: O diretório {caminho_diretorio} não foi encontrado.")
        return todas_musicas

    arquivos_docx = list(diretorio.glob("*.docx"))
    print(f"Encontrados {len(arquivos_docx)} arquivos .docx")

    for arquivo in arquivos_docx:
        print(f"Lendo arquivo: {arquivo.name}")
        doc = Document(arquivo)
        musica_atual = []

        for paragraph in doc.paragraphs:
            texto = paragraph.text.strip()
            if not texto:
                continue

            if re.search(r'\s+[-–]\s+', texto):
                if musica_atual:
                    todas_musicas.append('\n'.join(musica_atual))
                    musica_atual = []
            musica_atual.append(texto)

        if musica_atual:
            todas_musicas.append('\n'.join(musica_atual))

    return todas_musicas

In [ ]:
def limpar_texto(texto):
    """
    Remove marcações estruturais e símbolos indesejados,
    mantendo a estrutura de quebras de linha (\n) para correta separação em versos/estrofes, e preservando hifens/traços (- ou –) para identificação dos títulos.
    """
    # Remove colchetes, parênteses e conteúdos internos comuns de estruturas
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = re.sub(r'\(.*?\)', '', texto)

    # Remove marcações de repetição comuns, ex: "2x", "3x"
    texto = re.sub(r'\b\d+x\b', '', texto, flags=re.IGNORECASE)

    # Remove cabeçalhos de voz ou falas como "MC Davi:", "Mc Pedrinho:"
    texto = re.sub(r'(?i)\b(mc|voz|ponte|refrão|introdução|intro)\b.*?($|:)', '', texto)

    # Remove qualquer caractere que NÃO seja alfanumérico (letras, números), quebras de linha, espaços ou traços/hifens
    texto = re.sub(r'[^\b\u00C0-ÿ\u0100-\u017F\u0180-\u024F\u1E00-ỿ\u2C60-\u2C7F\uA720-\uA7FFa-zA-Z0-9\n\s\-\–\—]', '', texto)

    # Limpa múltiplos espaços horizontais por linha, preservando as quebras de linha
    linhas_limpas = []
    for linha in texto.splitlines():
        linha_limpa = re.sub(r'[^\u00C0-\u00FF\u0100-\u017F\u0180-\u024F\u1E00-\u1EFF\u2C60-\u2C7F\uA720-\uA7FFa-zA-Z0-9\u0020\-\–\—]', ' ', linha)
        linha_limpa = re.sub(r'\u0020+', ' ', linha_limpa).strip()
        if linha_limpa:
            linhas_limpas.append(linha_limpa)

    return '\n'.join(linhas_limpas)

In [ ]:
def analisar_textos_em_lote(lista_textos, batch_size=32):
    """
    Realiza a classificação de múltiplos textos em lote utilizando a GPU/CPU,
    evitando a lentidão de inferência de uma em uma.
    """
    resultados = []

    for i in range(0, len(lista_textos), batch_size):
        lote = lista_textos[i : i + batch_size]

        # Tokenização em lote com padding dinâmico
        entradas = tokenizer(
            lote,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding=True
        ).to(device)

        with torch.no_grad():
            saidas = modelo(**entradas)
            probabilidades = torch.softmax(saidas.logits, dim=-1)

        classes = torch.argmax(probabilidades, dim=-1).cpu().numpy()
        probs = probabilidades.cpu().numpy()

        for j, texto in enumerate(lote):
            classe = int(classes[j])
            resultados.append({
                "texto_limpo": texto,
                "classe": classe,
                "classificacao": "Misógino" if classe == 1 else "Não misógino",
                "prob_nao_misogino": float(probs[j][0]),
                "prob_misogino": float(probs[j][1])
            })

    return resultados

In [ ]:
def analisar_texto(texto):
    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        saida = modelo(**entradas)
        probabilidades = torch.softmax(saida.logits, dim=-1)[0]

    classe = int(torch.argmax(probabilidades).item())

    return {
        "classe": classe,
        "classificacao": "Misógino" if classe == 1 else "Não misógino",
        "prob_misogino": float(probabilidades[1])
    }

In [ ]:
def analisar_letra_por_estrofe(texto, titulo_musica):
    linhas = [linha.strip() for linha in texto.splitlines() if linha.strip()]

    # Agrupa de 4 em 4 linhas
    estrofes = ['\n'.join(linhas[i:i+4]) for i in range(0, len(linhas), 4)]

    # Chama a função de lote (Batch) JÁ DEFINIDA NO SEU CÓDIGO
    resultados_lote = analisar_textos_em_lote(estrofes, batch_size=8)

    # Formata a saída para o DataFrame
    resultados_formatados = []
    for numero, res in enumerate(resultados_lote, start=1):
        resultados_formatados.append({
            "estrofe_num": numero,
            "texto": res["texto_limpo"],
            "classificacao": res["classificacao"],
            "prob_misogino": res["prob_misogino"],
        })

    df = pd.DataFrame(resultados_formatados)
    return df

In [ ]:
def imprimir_arquivo_csv(df, nome_arquivo="resultados.csv"):
    df.to_csv(
        nome_arquivo, # Agora utiliza a variável em vez do texto fixo
        index=False,
        encoding="utf-8-sig",
        decimal=","
    )
    print(f"Arquivo {nome_arquivo} salvo com sucesso")

In [ ]:
pasta = Path("/content/drive/MyDrive/UNIVESP/Projeto integrador/PI4/PESQUISA MISOGINIA NO FUNK/Letras")

lista_musicas = extrair_musicas_de_diretorio(pasta)

lista_musicas_limpas = []
for musica in lista_musicas:
    lista_musicas_limpas.append(limpar_texto(musica))

resultados_completos = []
resultados_estrofes = []

# Percorre todas as músicas extraídas
for i, musica_texto in enumerate(lista_musicas_limpas):
    if len(musica_texto.strip()) < 10:
        continue

    # Identifica o título da música baseado nas primeiras linhas
    linhas = [l.strip() for l in musica_texto.splitlines() if l.strip()]
    titulo_identificado = f"Música {i}"
    if linhas:
        linha_titulo = None
        # Procura por uma linha que contenha traço/travessão nas 3 primeiras linhas para evitar pegar versos do meio
        for linha in linhas[:3]:
            if any(traco in linha for traco in ['-', '–', '—']):
                linha_titulo = linha
                break
        if linha_titulo:
            titulo_identificado = linha_titulo
        else:
            # Se a primeira linha for apenas o ano ou número, tenta usar a segunda
            if (linhas[0].isdigit() or len(linhas[0]) <= 4) and len(linhas) > 1:
                titulo_identificado = linhas[1]
            else:
                titulo_identificado = linhas[0]

    # 1. Análise da música inteira (Texto Completo)
    predicao_completa = analisar_texto(musica_texto)
    resultados_completos.append({
        "musica_id": i,
        "musica_titulo": titulo_identificado,
        "texto_completo": musica_texto,
        "analise_musica_completa": predicao_completa["classificacao"],
        "prob_misogino_completa": predicao_completa["prob_misogino"],
    })

    # 2. Análise da letra por estrofe
    df_musica_versos = analisar_letra_por_estrofe(musica_texto, titulo_identificado)
    df_musica_versos["musica_id"] = i
    df_musica_versos["musica_titulo"] = titulo_identificado
    resultados_estrofes.append(df_musica_versos)

# Criando o DataFrame apenas com os textos completos e análises globais
df_textos_completos = pd.DataFrame(resultados_completos)

# Criando o DataFrame consolidado com o detalhamento das estrofes
df_consolidado_estrofes = pd.concat(resultados_estrofes, ignore_index=True)

# Reordena as colunas do DataFrame de estrofes
colunas_ordenadas_estrofes = [
    'musica_id',
    'musica_titulo',
    'estrofe_num',
    'texto',
    'classificacao',
    'prob_misogino',
]
df_consolidado_estrofes = df_consolidado_estrofes[colunas_ordenadas_estrofes]

# Salva o resultado final unificado das estrofes em CSV
imprimir_arquivo_csv(df_consolidado_estrofes, "resultados_por_estrofes.csv" + modelo_nome)
imprimir_arquivo_csv(df_textos_completos, "resultados_texto_completo.csv" + modelo_nome)